# ECG Heartbeat Classification: CNN vs CNN-LSTM vs Transformer

**Author:** Poonam Sharma  
**Course:** Deep Learning & Artificial Intelligence  
**Dataset:** MIT-BIH Arrhythmia Database

This notebook walks through the project, from data exploration to model evaluation.
All models are trained by running `python main.py` from the project root.
This notebook assumes preprocessing and training have already been completed.

In [ ]:
import sys
import os
sys.path.insert(0, '..')  # so we can import project modules

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter
import torch

DATA_DIR    = '../data'
MODELS_DIR  = '../models'
RESULTS_DIR = '../results'

CLASS_NAMES = ['N (Normal)', 'S (SVE)', 'V (VE)', 'F (Fusion)', 'Q (Unknown)']
print('Imports OK')

## 1. Dataset Overview

In [ ]:
# load preprocessed splits
X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

print(f'Train : {X_train.shape}  (after SMOTE balancing)')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

# class distribution in test set (original, no SMOTE)
test_counts = Counter(y_test)
print('\nTest set class distribution:')
for cls, name in enumerate(CLASS_NAMES):
    n = test_counts[cls]
    print(f'  {name}: {n:,} ({100*n/len(y_test):.1f}%)')

In [ ]:
# bar chart of test set class imbalance
fig, ax = plt.subplots(figsize=(8, 4))
counts  = [test_counts[i] for i in range(5)]
colors  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
bars    = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor='white')
ax.set_ylabel('Number of Beats')
ax.set_title('MIT-BIH Test Set Class Distribution (Original, No SMOTE)')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Sample ECG Beat Waveforms

Each beat is a 180-sample window (500 ms at 360 Hz) centered on the R-peak.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
t = np.linspace(0, 500, 180)  # milliseconds

for cls in range(5):
    idx = np.where(y_test == cls)[0][0]  # first sample of this class
    axes[cls].plot(t, X_test[idx], color=colors[cls], linewidth=1.2)
    axes[cls].set_title(CLASS_NAMES[cls], fontsize=10)
    axes[cls].set_xlabel('ms')
    axes[cls].grid(True, alpha=0.3)

axes[0].set_ylabel('Amplitude (normalized)')
fig.suptitle('Sample ECG Beat from Each AAMI Class', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Model Architectures

In [ ]:
from models.cnn         import PlainCNN
from models.heartnet    import HeartNet
from models.transformer import ECGTransformer

models = [
    ('Plain CNN',   PlainCNN(5)),
    ('HeartNet',    HeartNet(5)),
    ('Transformer', ECGTransformer(5)),
]

print(f"{'Model':<20} {'Parameters':>15}")
print('-' * 36)
for name, m in models:
    n = sum(p.numel() for p in m.parameters())
    print(f'{name:<20} {n:>15,}')

# verify forward pass shapes
dummy = torch.randn(4, 1, 180)
for name, m in models:
    out = m(dummy)
    assert out.shape == (4, 5), f'{name} output shape mismatch!'
print('\nForward pass check passed for all models.')

## 4. Evaluation Results

In [ ]:
from evaluate import evaluate_model

configs = [
    (PlainCNN,       'Plain_CNN',   os.path.join(MODELS_DIR, 'plain_cnn_best.pth')),
    (HeartNet,       'HeartNet',    os.path.join(MODELS_DIR, 'heartnet_best.pth')),
    (ECGTransformer, 'Transformer', os.path.join(MODELS_DIR, 'transformer_best.pth')),
]

results = []
for ModelClass, name, ckpt in configs:
    if not os.path.exists(ckpt):
        print(f'Checkpoint not found for {name} — run python main.py first.')
        continue
    r = evaluate_model(ModelClass(5), name, ckpt, X_test, y_test)
    results.append(r)

In [ ]:
# summary table
if results:
    print(f"{'Model':<18} {'Accuracy':>10} {'F1':>8} {'Recall':>8} {'AUC-ROC':>9}")
    print('-' * 55)
    for r in results:
        print(f"{r['model']:<18} {r['accuracy']:>10.4f} {r['f1']:>8.4f} "
              f"{r['recall']:>8.4f} {r['auc_roc']:>9.4f}")

## 5. Saved Plots

In [ ]:
# training curves
for fname in ['plain_cnn_training_curves.png', 'heartnet_training_curves.png']:
    path = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'{fname} not found — run main.py first.')

In [ ]:
# confusion matrices
cm_files = ['plain_cnn_confusion_matrix.png',
            'heartnet_confusion_matrix.png',
            'transformer_confusion_matrix.png']

available = [f for f in cm_files if os.path.exists(os.path.join(RESULTS_DIR, f))]
if available:
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, fname in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(RESULTS_DIR, fname)))
        ax.axis('off')
        ax.set_title(fname.replace('_confusion_matrix.png', '').replace('_', ' ').title())
    plt.tight_layout()
    plt.show()
else:
    print('No confusion matrix PNGs found — run main.py first.')

In [ ]:
# ROC curves
roc_path = os.path.join(RESULTS_DIR, 'heartnet_roc_curves.png')
if os.path.exists(roc_path):
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(mpimg.imread(roc_path))
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('ROC curves PNG not found — run main.py first.')